# 3-Model Ensemble Validation
## Reference Period: 1995–2014 | SSP2-4.5

**Purpose:**  
Evaluate the performance of the basin-specific 3-model ensemble against observed precipitation climatology using the same five statistical metrics and composite ranking framework applied to individual models in Notebook 03. Results can be directly compared with single-model performance to assess whether the selective ensemble approach improves upon or degrades the best single-model skill.

**Evaluation approach** 
1. Compare 12-month observed LTMM vs ensemble LTMM at each station  
2. Compute five metrics per station: r, NSE, PBIAS, RMSE, MAE  
3. Average station metrics to basin level  
4. Produce a summary table parallel to Table 3 for the ensemble  
5. Direct numerical comparison: ensemble vs best single model per basin

**Metrics reference:**

| Metric | Optimal | Direction |
|--------|---------|-----------|
| Pearson's r | 1.0 | Higher is better |
| NSE | 1.0 | Higher is better |
| PBIAS (%) | 0.0 | Closer to zero is better |
| RMSE (mm/month) | 0.0 | Lower is better |
| MAE (mm/month) | 0.0 | Lower is better |

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.3model\`


## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

print("Libraries loaded.")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")


Libraries loaded.
  numpy  : 2.1.3
  pandas : 2.2.3


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
# Observed LTMM — from Notebook 02
OBS_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv"
)

# Ensemble LTMM — from Notebook 06
ENS_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\ensemble data\long_term_monthly_mean\ltmm_ensemble_all_stations.csv"
)

# Single-model validation results — from Notebook 03 (for comparison)
SINGLE_TABLE3_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\single.model\table3_best_model_per_basin.csv"
)

SINGLE_BASIN_METRICS_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\single.model\basin_level_metrics.csv"
)

# Station mapping
STATION_MAPPING_FILE = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\ensemble.3model"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Obs LTMM      : {OBS_LTMM_CSV}")
print(f"  Ensemble LTMM : {ENS_LTMM_CSV}")
print(f"  Output        : {OUTPUT_ROOT}")


Configuration loaded.
  Obs LTMM      : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv
  Ensemble LTMM : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data\long_term_monthly_mean\ltmm_ensemble_all_stations.csv
  Output        : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.3model


## 3. Statistical Performance Metrics

Identical functions to Notebook 03 — reproduced here for self-contained execution without dependency on the earlier notebook's kernel state.

| Metric | Formula | Optimal |
|--------|---------|---------|
| **r** | $\frac{\sum(O-\bar{O})(M-\bar{M})}{\sqrt{\sum(O-\bar{O})^2\sum(M-\bar{M})^2}}$ | 1.0 |
| **NSE** | $1 - \frac{\sum(O-M)^2}{\sum(O-\bar{O})^2}$ | 1.0 |
| **PBIAS** | $\frac{\sum(M-O)}{\sum O}\times 100$ | 0.0 |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(M-O)^2}$ | 0.0 |
| **MAE** | $\frac{1}{n}\sum|M-O|$ | 0.0 |


In [3]:
def compute_metrics(obs: np.ndarray, mod: np.ndarray) -> dict:
    """
    Compute five evaluation metrics between observed and modelled arrays.
    Pairs containing NaN are excluded before computation.

    Parameters
    ----------
    obs : array-like of observed values
    mod : array-like of modelled values

    Returns
    -------
    dict  with keys r, NSE, PBIAS, RMSE, MAE, N
    """
    obs = np.asarray(obs, dtype=float)
    mod = np.asarray(mod, dtype=float)
    mask = np.isfinite(obs) & np.isfinite(mod)
    obs, mod = obs[mask], mod[mask]
    n = len(obs)

    if n < 3:
        return {"r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
                "RMSE": np.nan, "MAE": np.nan, "N": n}

    obs_mean = np.mean(obs)

    num   = np.sum((obs - obs_mean) * (mod - np.mean(mod)))
    denom = np.sqrt(np.sum((obs - obs_mean)**2) * np.sum((mod - np.mean(mod))**2))
    r     = num / denom if denom > 0 else np.nan

    ss_res = np.sum((obs - mod)**2)
    ss_tot = np.sum((obs - obs_mean)**2)
    nse    = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

    pbias  = (np.sum(mod - obs) / np.sum(obs)) * 100 if np.sum(obs) != 0 else np.nan
    rmse   = np.sqrt(np.mean((mod - obs)**2))
    mae    = np.mean(np.abs(mod - obs))

    return {"r": round(r, 6), "NSE": round(nse, 6),
            "PBIAS": round(pbias, 4), "RMSE": round(rmse, 4),
            "MAE": round(mae, 4), "N": n}


print("Metric functions defined: r | NSE | PBIAS | RMSE | MAE")


Metric functions defined: r | NSE | PBIAS | RMSE | MAE


## 4. Load Observed and Ensemble LTMM Data


In [4]:
# ── Observed LTMM ─────────────────────────────────────────────────────────────
obs_ltmm = pd.read_csv(OBS_LTMM_CSV)
obs_ltmm["Station_ID"] = obs_ltmm["Station_ID"].astype(str).str.strip()
print(f"Observed LTMM : {len(obs_ltmm):,} rows  "
      f"({obs_ltmm['Station_ID'].nunique()} stations × 12 months)")

# ── Ensemble LTMM ─────────────────────────────────────────────────────────────
ens_ltmm = pd.read_csv(ENS_LTMM_CSV)
ens_ltmm["Station_ID"] = ens_ltmm["Station_ID"].astype(str).str.strip()
print(f"Ensemble LTMM : {len(ens_ltmm):,} rows  "
      f"({ens_ltmm['Station_ID'].nunique()} stations × 12 months)")

# ── Station mapping for basin order ──────────────────────────────────────────
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"] = stations["Station_ID"].astype(str).str.strip()
stations["Basin"]      = stations["Basin"].astype(str).str.strip()
basin_order = list(dict.fromkeys(stations["Basin"].tolist()))

print(f"\nBasins in validation order ({len(basin_order)}):")
for b in basin_order:
    print(f"  {b}")


Observed LTMM : 588 rows  (49 stations × 12 months)
Ensemble LTMM : 588 rows  (49 stations × 12 months)

Basins in validation order (12):
  N.R.S.W
  YARMOUK (JORDAN)
  JORDAN VALLY (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  D.S.R.S.W
  MUJIB
  W. ARABA NORTH
  HASA
  AZRAQ (JORDAN)
  JAFER
  HAMMAD


## 5. Station-Level Performance Metrics

For each of the 49 stations, the 12-month observed LTMM vector is compared against the 12-month ensemble LTMM vector. This gives one metric set per station — 49 total.


In [5]:
ENSEMBLE_LABEL = "3-Model Ensemble"
station_metric_rows = []
obs_stations = obs_ltmm["Station_ID"].unique()

for sid in obs_stations:
    # Observed 12-month vector
    obs_vec = (obs_ltmm[obs_ltmm["Station_ID"] == sid]
               .sort_values("Month")
               .set_index("Month")["LTMM_mm_month"])

    # Ensemble 12-month vector
    ens_vec = (ens_ltmm[ens_ltmm["Station_ID"] == sid]
               .sort_values("Month")
               .set_index("Month")["LTMM_mm_month"])

    # Align on month (inner join)
    aligned = obs_vec.to_frame("obs").join(ens_vec.to_frame("mod"), how="inner")
    if aligned.empty or len(aligned) < 3:
        continue

    metrics = compute_metrics(aligned["obs"].values, aligned["mod"].values)

    # Basin and name from obs table
    basin = obs_ltmm.loc[obs_ltmm["Station_ID"] == sid, "Basin"].iloc[0]
    sname = obs_ltmm.loc[obs_ltmm["Station_ID"] == sid, "Station_Name"].iloc[0]

    station_metric_rows.append({
        "Model"        : ENSEMBLE_LABEL,
        "Station_ID"   : sid,
        "Station_Name" : sname,
        "Basin"        : basin,
        **metrics
    })

stn_metrics_df = pd.DataFrame(station_metric_rows)
stn_metrics_df.to_csv(OUTPUT_ROOT / "station_level_metrics_ensemble.csv", index=False)

print(f"Station metrics computed : {len(stn_metrics_df)} stations")
print(f"Saved: station_level_metrics_ensemble.csv")
print()
print("Overall ensemble statistics (mean across all stations):")
print(stn_metrics_df[["r","NSE","PBIAS","RMSE","MAE"]].mean().round(4).to_string())


Station metrics computed : 49 stations
Saved: station_level_metrics_ensemble.csv

Overall ensemble statistics (mean across all stations):
r         0.9704
NSE       0.4763
PBIAS    16.9482
RMSE      9.7588
MAE       6.1669


## 6. Aggregate to Basin-Level Metrics

Station-level metrics are averaged across all stations within each basin, consistent with the approach in Notebook 03.


In [6]:
basin_metric_rows = []

for basin in basin_order:
    grp = stn_metrics_df[stn_metrics_df["Basin"] == basin]

    if grp.empty:
        basin_metric_rows.append({
            "Basin": basin, "Model": ENSEMBLE_LABEL,
            "r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
            "RMSE": np.nan, "MAE": np.nan, "N_Stations": 0
        })
        continue

    basin_metric_rows.append({
        "Basin"      : basin,
        "Model"      : ENSEMBLE_LABEL,
        "r"          : round(grp["r"].mean(),    6),
        "NSE"        : round(grp["NSE"].mean(),  6),
        "PBIAS"      : round(grp["PBIAS"].mean(),4),
        "RMSE"       : round(grp["RMSE"].mean(), 4),
        "MAE"        : round(grp["MAE"].mean(),  4),
        "N_Stations" : len(grp),
    })

basin_metrics_df = pd.DataFrame(basin_metric_rows)
basin_metrics_df.to_csv(OUTPUT_ROOT / "basin_level_metrics_ensemble.csv", index=False)

print("Basin-level ensemble metrics:")
print(f"Saved: basin_level_metrics_ensemble.csv")
print()
print(basin_metrics_df[["Basin","r","NSE","PBIAS","RMSE","MAE","N_Stations"]]
      .to_string(index=False))


Basin-level ensemble metrics:
Saved: basin_level_metrics_ensemble.csv

                Basin        r       NSE    PBIAS    RMSE     MAE  N_Stations
              N.R.S.W 0.984864  0.902317  -7.0458 12.8168  7.6028           4
     YARMOUK (JORDAN) 0.981598  0.598058  -8.3617 10.5690  6.7468           5
JORDAN VALLY (JORDAN) 0.991970  0.107565  -2.4682 23.0201 15.6716           3
 AMMAN ZARQA (JORDAN) 0.985762  0.457294  33.7484 10.9467  7.1467           6
              S.R.S.W 0.991959  0.737769  -3.1602 15.0550  9.2205           4
            D.S.R.S.W 0.978087  0.911733  -9.3710  7.6683  4.4995           7
                MUJIB 0.971398  0.532823  28.0879  7.7968  4.9571           6
       W. ARABA NORTH 0.937587  0.796515 -17.2847  9.3498  5.0618           3
                 HASA 0.959779  0.903139  -6.2377  5.2912  3.2050           1
       AZRAQ (JORDAN) 0.931212 -0.870272 107.1386  6.0097  4.3574           4
                JAFER 0.960356  0.194357  38.9437  3.4224  2.2777      

## 7. Ensemble Summary Table

A basin-level summary table, showing the 3-model ensemble metrics for each basin. Mean and standard deviation rows are appended for easy comparison.


In [7]:
table_ens = basin_metrics_df[["Basin","r","NSE","PBIAS","RMSE","MAE"]].copy()

numeric_cols = ["r","NSE","PBIAS","RMSE","MAE"]
mean_row = {"Basin": "Mean"}
std_row  = {"Basin": "Std Dev"}
for col in numeric_cols:
    vals = pd.to_numeric(table_ens[col], errors="coerce").dropna()
    mean_row[col] = round(vals.mean(), 3)
    std_row[col]  = round(vals.std(),  3)

table_ens_full = pd.concat(
    [table_ens, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

table_ens_full.to_csv(OUTPUT_ROOT / "table_ensemble_basin_metrics.csv", index=False)

# ── Excel formatted version ───────────────────────────────────────────────────
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb  = Workbook()
    ws  = wb.active
    ws.title = "Ensemble Basin Metrics"

    thin   = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left   = Alignment(horizontal="left",   vertical="center")
    title_font = Font(name="Arial", bold=True, size=12, color="1F4E79")
    hdr_fill   = PatternFill("solid", start_color="1F4E79")
    mean_fill  = PatternFill("solid", start_color="EBF5FB")
    std_fill   = PatternFill("solid", start_color="F8F9FA")

    ws.merge_cells("A1:F1")
    ws["A1"].value     = "3-Model Ensemble Performance Metrics Across Jordan Basins (1995-2014)"
    ws["A1"].font      = title_font
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 22

    headers    = ["Basin", "Correlation (r)", "NSE", "PBIAS (%)", "RMSE (mm/month)", "MAE (mm/month)"]
    col_widths = [24, 14, 10, 12, 16, 16]
    for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
        c = ws.cell(row=2, column=ci, value=h)
        c.fill      = hdr_fill
        c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=9)
        c.alignment = center
        c.border    = border
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[2].height = 32

    for ri, row in table_ens_full.iterrows():
        r         = ri + 3
        is_stat   = row["Basin"] in ("Mean", "Std Dev")
        row_fill  = mean_fill if row["Basin"] == "Mean" else (
                    std_fill  if row["Basin"] == "Std Dev" else None)
        vals = [row["Basin"], row["r"], row["NSE"], row["PBIAS"], row["RMSE"], row["MAE"]]
        for ci, v in enumerate(vals, 1):
            cell = ws.cell(row=r, column=ci)
            cell.value     = v if not (isinstance(v, float) and np.isnan(v)) else ""
            cell.font      = Font(name="Arial", bold=is_stat, size=9)
            cell.alignment = left if ci == 1 else center
            cell.border    = border
            if row_fill:
                cell.fill  = row_fill
            if ci > 1 and isinstance(v, float) and not np.isnan(v):
                cell.number_format = "0.000"

    ws.freeze_panes = "A3"
    xl_path = OUTPUT_ROOT / "table_ensemble_basin_metrics.xlsx"
    wb.save(xl_path)
    print(f"Excel table saved: {xl_path}")
except Exception as e:
    print(f"Excel formatting skipped: {e}")

print()
print("3-Model Ensemble Basin Metrics:")
print("=" * 65)
print(table_ens_full.to_string(index=False))


Excel table saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\ensemble.3model\table_ensemble_basin_metrics.xlsx

3-Model Ensemble Basin Metrics:
                Basin        r       NSE    PBIAS    RMSE     MAE
              N.R.S.W 0.984864  0.902317  -7.0458 12.8168  7.6028
     YARMOUK (JORDAN) 0.981598  0.598058  -8.3617 10.5690  6.7468
JORDAN VALLY (JORDAN) 0.991970  0.107565  -2.4682 23.0201 15.6716
 AMMAN ZARQA (JORDAN) 0.985762  0.457294  33.7484 10.9467  7.1467
              S.R.S.W 0.991959  0.737769  -3.1602 15.0550  9.2205
            D.S.R.S.W 0.978087  0.911733  -9.3710  7.6683  4.4995
                MUJIB 0.971398  0.532823  28.0879  7.7968  4.9571
       W. ARABA NORTH 0.937587  0.796515 -17.2847  9.3498  5.0618
                 HASA 0.959779  0.903139  -6.2377  5.2912  3.2050
       AZRAQ (JORDAN) 0.931212 -0.870272 107.1386  6.0097  4.3574
                JAFER 0.960356  0.194357  38.9437  3.4224  2.2777
               HAMMAD 0.900202  0.359861  

## 8. Direct Comparison: 3-Model Ensemble vs Best Single Model

The ensemble basin metrics are compared side-by-side with the best single-model results from Notebook 03. For each metric, the difference (Ensemble − Best Single) is computed: negative values for RMSE/MAE/|PBIAS| indicate ensemble improvement; positive values indicate degradation. For r and NSE, positive differences indicate improvement.

This comparison directly evaluates the ensemble approach relative to selective single-model use — a central analytical question in the manuscript.


In [8]:
# Load single-model best results
single_t3 = pd.read_csv(SINGLE_TABLE3_CSV)
# Keep only data rows (exclude Mean/Std Dev summary rows)
single_data = single_t3[~single_t3["Basin"].isin(["Mean","Std Dev"])].copy()

# Load single-model ALL models basin metrics for context
single_basin_all = pd.read_csv(SINGLE_BASIN_METRICS_CSV)

# Merge ensemble metrics with single best
compare = basin_metrics_df[["Basin","r","NSE","PBIAS","RMSE","MAE"]].copy()
compare = compare.rename(columns={
    "r": "Ens_r", "NSE": "Ens_NSE", "PBIAS": "Ens_PBIAS",
    "RMSE": "Ens_RMSE", "MAE": "Ens_MAE"
})

# Merge with single-model best (from table3)
single_sub = single_data[["Basin","Best_Model","r","NSE","PBIAS","RMSE","MAE"]].rename(columns={
    "r": "Best_r", "NSE": "Best_NSE", "PBIAS": "Best_PBIAS",
    "RMSE": "Best_RMSE", "MAE": "Best_MAE"
})

compare = compare.merge(single_sub, on="Basin", how="left")

# Compute differences: Ensemble - Best Single
# For r, NSE: positive = ensemble better
# For RMSE, MAE, |PBIAS|: negative = ensemble better
compare["Delta_r"]          = (compare["Ens_r"]       - compare["Best_r"]).round(4)
compare["Delta_NSE"]        = (compare["Ens_NSE"]     - compare["Best_NSE"]).round(4)
compare["Delta_PBIAS"]      = (compare["Ens_PBIAS"].abs() - compare["Best_PBIAS"].abs()).round(4)
compare["Delta_RMSE"]       = (compare["Ens_RMSE"]   - compare["Best_RMSE"]).round(4)
compare["Delta_MAE"]        = (compare["Ens_MAE"]    - compare["Best_MAE"]).round(4)

compare["RMSE_improved"]    = compare["Delta_RMSE"]  < 0
compare["MAE_improved"]     = compare["Delta_MAE"]   < 0
compare["r_improved"]       = compare["Delta_r"]     > 0
compare["NSE_improved"]     = compare["Delta_NSE"]   > 0

compare.to_csv(OUTPUT_ROOT / "ensemble_vs_best_single_comparison.csv", index=False)

print("DIRECT COMPARISON: 3-Model Ensemble vs Best Single Model per Basin")
print("=" * 90)
print(f"{'Basin':<28} {'Best Model':<22} {'Delt_r':>7} {'Delt_NSE':>9} "
      f"{'Delt_RMSE':>10} {'Delt_MAE':>9} {'Delt_|PBIAS|':>12}")
print("-" * 90)
for _, row in compare.iterrows():
    bm = str(row.get("Best_Model", "?"))
    print(f"{row['Basin']:<28} {bm:<22} "
          f"{row['Delta_r']:>+7.4f} {row['Delta_NSE']:>+9.4f} "
          f"{row['Delta_RMSE']:>+10.4f} {row['Delta_MAE']:>+9.4f} "
          f"{row['Delta_PBIAS']:>+12.4f}")

print()
print("Summary (basins where ensemble improves over best single model):")
for metric, col in [("r", "r_improved"), ("NSE", "NSE_improved"),
                     ("RMSE", "RMSE_improved"), ("MAE", "MAE_improved")]:
    n = compare[col].sum()
    print(f"  {metric:<8}: ensemble better in {n}/{len(compare)} basins")

print()
print("Saved: ensemble_vs_best_single_comparison.csv")


DIRECT COMPARISON: 3-Model Ensemble vs Best Single Model per Basin
Basin                        Best Model              Delt_r  Delt_NSE  Delt_RMSE  Delt_MAE Delt_|PBIAS|
------------------------------------------------------------------------------------------
N.R.S.W                      NorESM2-MM             +0.0010   +0.0002    +0.0245   -0.0089      +0.2740
YARMOUK (JORDAN)             MPI-ESM1-2-LR          +0.0167   -0.2104    +1.6085   +1.3075     -11.8256
JORDAN VALLY (JORDAN)        MPI-ESM1-2-LR          +0.0048   -0.5409    +8.2836   +5.8449     -37.9567
AMMAN ZARQA (JORDAN)         MPI-ESM1-2-LR          +0.0015   -0.0363    +0.1155   +0.1818      +4.4193
S.R.S.W                      MPI-ESM1-2-LR          -0.0032   -0.0221    +0.2976   -0.0659      -3.3351
D.S.R.S.W                    IPSL-CM6A-LR           +0.0111   +0.0063    -0.3828   -0.4731      +5.6203
MUJIB                        MPI-ESM1-2-LR          -0.0019   -0.0575    +0.3074   +0.1550      +6.0345
W. ARABA N

## 9. Median Metric Summary: Ensemble vs All Three Approaches

A final summary table showing median metric values across all basins for three approaches: best single model, 3-model ensemble (this notebook), and the 6-model ensemble (from Notebook 03).


In [9]:
# Load the 3-approach comparison from NB03 (if available)
approach_csv = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\single.model\ensemble_comparison_basin.csv"
)

summary_rows = []

# Best single model (from NB03 table3)
if SINGLE_TABLE3_CSV.exists():
    t3 = pd.read_csv(SINGLE_TABLE3_CSV)
    t3_data = t3[~t3["Basin"].isin(["Mean","Std Dev"])]
    summary_rows.append({
        "Approach"    : "Best Single Model",
        "Median_r"    : round(t3_data["r"].median(),    3),
        "Median_NSE"  : round(t3_data["NSE"].median(),  3),
        "Median_RMSE" : round(t3_data["RMSE"].median(), 3),
        "Median_MAE"  : round(t3_data["MAE"].median(),  3),
        "Median_PBIAS": round(t3_data["PBIAS"].abs().median(), 3),
    })

# 3-Model Ensemble (this notebook)
ens_data = basin_metrics_df[pd.to_numeric(basin_metrics_df["r"], errors="coerce").notna()]
summary_rows.append({
    "Approach"    : "3-Model Ensemble (basin-specific)",
    "Median_r"    : round(ens_data["r"].median(),    3),
    "Median_NSE"  : round(ens_data["NSE"].median(),  3),
    "Median_RMSE" : round(ens_data["RMSE"].median(), 3),
    "Median_MAE"  : round(ens_data["MAE"].median(),  3),
    "Median_PBIAS": round(ens_data["PBIAS"].abs().median(), 3),
})

# 6-model ensemble from NB03 if available
if approach_csv.exists():
    ap = pd.read_csv(approach_csv)
    for approach_label in ["3-Model Ensemble", "6-Model Ensemble"]:
        sub = ap[ap["Approach"] == approach_label]
        if not sub.empty:
            summary_rows.append({
                "Approach"    : f"{approach_label} (uniform, NB03)",
                "Median_r"    : round(sub["r"].median(),    3),
                "Median_NSE"  : round(sub["NSE"].median(),  3),
                "Median_RMSE" : round(sub["RMSE"].median(), 3),
                "Median_MAE"  : round(sub["MAE"].median(),  3),
                "Median_PBIAS": round(sub["PBIAS"].abs().median(), 3),
            })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "approach_median_comparison.csv", index=False)

print("MEDIAN METRIC COMPARISON ACROSS APPROACHES")
print("=" * 80)
print(summary_df.to_string(index=False))
print()
print("Saved: approach_median_comparison.csv")
print()
print("Note on metrics:")
print("  r, NSE    : higher is better (closer to 1.0)")
print("  RMSE, MAE : lower is better (closer to 0.0)")
print("  |PBIAS|   : lower is better (closer to 0.0)")


MEDIAN METRIC COMPARISON ACROSS APPROACHES
                         Approach  Median_r  Median_NSE  Median_RMSE  Median_MAE  Median_PBIAS
                Best Single Model     0.970       0.704        8.506       5.172        21.120
3-Model Ensemble (basin-specific)     0.975       0.565        8.573       5.009        13.328
 3-Model Ensemble (uniform, NB03)     0.932       0.813        8.651       5.015        15.508
 6-Model Ensemble (uniform, NB03)     0.919       0.790        9.367       5.193        16.697

Saved: approach_median_comparison.csv

Note on metrics:
  r, NSE    : higher is better (closer to 1.0)
  RMSE, MAE : lower is better (closer to 0.0)
  |PBIAS|   : lower is better (closer to 0.0)


## 10. Output Summary

In [10]:
print("=" * 72)
print("ENSEMBLE VALIDATION OUTPUT FILES")
print("=" * 72)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root) / f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("=" * 72)
print("VALIDATION COMPLETE")
print("=" * 72)
print()
print("Output files:")
print("  station_level_metrics_ensemble.csv   — 49 stations x 5 metrics")
print("  basin_level_metrics_ensemble.csv     — basin-averaged metrics")
print("  table_ensemble_basin_metrics.csv     — summary table (parallel to Table 3)")
print("  table_ensemble_basin_metrics.xlsx    — formatted Excel version")
print("  ensemble_vs_best_single_comparison.csv — delta metrics per basin")
print("  approach_median_comparison.csv       — median across all approaches")


ENSEMBLE VALIDATION OUTPUT FILES
📁 ensemble.3model/
  📄 approach_median_comparison.csv  (0.3 KB)
  📄 basin_level_metrics_ensemble.csv  (0.9 KB)
  📄 ensemble_vs_best_single_comparison.csv  (2.1 KB)
  📄 station_level_metrics_ensemble.csv  (4.5 KB)
  📄 table_ensemble_basin_metrics.csv  (0.7 KB)
  📄 table_ensemble_basin_metrics.xlsx  (6.0 KB)

VALIDATION COMPLETE

Output files:
  station_level_metrics_ensemble.csv   — 49 stations x 5 metrics
  basin_level_metrics_ensemble.csv     — basin-averaged metrics
  table_ensemble_basin_metrics.csv     — summary table (parallel to Table 3)
  table_ensemble_basin_metrics.xlsx    — formatted Excel version
  ensemble_vs_best_single_comparison.csv — delta metrics per basin
  approach_median_comparison.csv       — median across all approaches
